In [1]:
import pyspark
from pyspark.sql import SparkSession

## DEFINE SENSITIVE VARIABLES
CATALOG_URI = "http://catalog:19120/api/v1"  # Nessie Server URI
WAREHOUSE = "s3://nessie-bucket/warehouse"               # Minio Address to Write to
STORAGE_URI = "http://minio:9000"      # Minio URL

In [2]:
"""### !!!!! PACKAGES WILL BE DOWNLOAD ONLINE
# Configure Spark with necessary packages and Iceberg/Nessie settings
conf = (
    pyspark.SparkConf()
        .setAppName('Nessie-Iceberg-Client')
        # Include necessary packages
        .set('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.99.0,software.amazon.awssdk:bundle:2.24.8,software.amazon.awssdk:url-connection-client:2.24.8')
        # Enable Iceberg and Nessie extensions
        .set('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
        # Configure Nessie catalog
        .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
        .set('spark.sql.catalog.nessie.uri', CATALOG_URI)
        .set('spark.sql.catalog.nessie.ref', 'main')
        .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
        .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
        .set("spark.sql.catalog.nessie.lock-manager-impl", "org.apache.iceberg.aws.s3.S3LockManager")
        .set("spark.sql.catalog.nessie.s3.lock.table", "iceberg_lock_table")
        # Set Minio as the S3 endpoint for Iceberg storage
        .set('spark.sql.catalog.nessie.s3.endpoint', STORAGE_URI)
        .set('spark.sql.catalog.nessie.warehouse', WAREHOUSE)
        .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
        .set("spark.sql.catalog.nessie.s3.path-style-access", "true")
        # Optimization
        .set("spark.sql.iceberg.read.split.target-size", "131072")
        .set("spark.executor.memory", "800m")
        .set("spark.executor.cores", "1")
        .set("spark.sql.adaptive.enabled", "true")
        .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .set("spark.sql.iceberg.vectorization.enabled", "false")
        .set("spark.sql.iceberg.distribution-mode", "hash")
        .set("spark.sql.execution.arrow.pyspark.enabled", "true")
)
"""

### !!!! PACKAGES ARE MOUNT IN FOLDER "/home/jovyan/jars"
import os
# Liste des JARs locaux
jars_path = "/home/jovyan/jars"
local_jars = ",".join([os.path.join(jars_path, f) for f in os.listdir(jars_path) if f.endswith('.jar')])
print(local_jars)
# Configure Spark with necessary packages and Iceberg/Nessie settings
conf = (
    pyspark.SparkConf()
        .setAppName('Nessie-Iceberg-Client')
        .set("spark.master", "spark://spark-master:7077") \
        # load local jar files
        .set("spark.jars", local_jars)
        # Forcer Spark à ne pas chercher sur Internet (Ivy offline)
        .set("spark.io.network.timeout", "60s")
        .set("spark.submit.deployMode", "client")
        .set("spark.jars.ivy.settings", "/dev/null")
        .set("spark.pyspark.python", "python3")
        # Include necessary packages
        #.set('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.99.0,software.amazon.awssdk:bundle:2.24.8,software.amazon.awssdk:url-connection-client:2.24.8')
        # Enable Iceberg and Nessie extensions
        .set('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
        # Configure Nessie catalog
        .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
        .set('spark.sql.catalog.nessie.uri', CATALOG_URI)
        .set('spark.sql.catalog.nessie.ref', 'main')
        .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
        .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
        .set("spark.sql.catalog.nessie.lock-manager-impl", "org.apache.iceberg.aws.s3.S3LockManager")
        .set("spark.sql.catalog.nessie.s3.lock.table", "iceberg_lock_table")
        # Set Minio as the S3 endpoint for Iceberg storage
        .set('spark.sql.catalog.nessie.s3.endpoint', STORAGE_URI)
        .set('spark.sql.catalog.nessie.warehouse', WAREHOUSE)
        .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
        .set("spark.sql.catalog.nessie.s3.path-style-access", "true")
        # Optimization
        .set("spark.sql.iceberg.read.split.target-size", "131072")
        .set("spark.executor.memory", "800m")
        .set("spark.executor.cores", "1")
        .set("spark.sql.adaptive.enabled", "true")
        .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .set("spark.sql.iceberg.vectorization.enabled", "false")
        .set("spark.sql.iceberg.distribution-mode", "hash")
        .set("spark.sql.execution.arrow.pyspark.enabled", "true")
)


/home/jovyan/jars/org.reactivestreams_reactive-streams-1.0.4.jar,/home/jovyan/jars/software.amazon.awssdk_metrics-spi-2.24.8.jar,/home/jovyan/jars/software.amazon.awssdk_bundle-2.24.8.jar,/home/jovyan/jars/org.projectnessie.nessie-integrations_nessie-spark-extensions-3.4_2.12-0.99.0.jar,/home/jovyan/jars/org.apache.iceberg_iceberg-spark-runtime-3.4_2.12-1.3.1.jar,/home/jovyan/jars/software.amazon.awssdk_url-connection-client-2.24.8.jar,/home/jovyan/jars/org.slf4j_slf4j-api-1.7.30.jar,/home/jovyan/jars/software.amazon.awssdk_utils-2.24.8.jar,/home/jovyan/jars/software.amazon.awssdk_http-client-spi-2.24.8.jar,/home/jovyan/jars/software.amazon.awssdk_annotations-2.24.8.jar


In [3]:
# Start Spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("Spark Session Started")

Spark Session Started


In [7]:
# check if all jars are available
print("----- Downloaded -----")
!ls /home/jovyan/.ivy2/jars
print("----- Local -----")
!ls /home/jovyan/jars

----- Downloaded -----
ls: cannot access '/home/jovyan/.ivy2/jars': No such file or directory
----- Local -----
org.apache.iceberg_iceberg-spark-runtime-3.4_2.12-1.3.1.jar
org.projectnessie.nessie-integrations_nessie-spark-extensions-3.4_2.12-0.99.0.jar
org.reactivestreams_reactive-streams-1.0.4.jar
org.slf4j_slf4j-api-1.7.30.jar
software.amazon.awssdk_annotations-2.24.8.jar
software.amazon.awssdk_bundle-2.24.8.jar
software.amazon.awssdk_http-client-spi-2.24.8.jar
software.amazon.awssdk_metrics-spi-2.24.8.jar
software.amazon.awssdk_url-connection-client-2.24.8.jar
software.amazon.awssdk_utils-2.24.8.jar


In [9]:
# check size of packages
print("----- Downloaded -----")
!du -hs /home/jovyan/.ivy2/jars/*
print("----- local -----")
!du -hs /home/jovyan/jars/*

----- Downloaded -----
du: cannot access '/home/jovyan/.ivy2/jars/*': No such file or directory
----- local -----
28M	/home/jovyan/jars/org.apache.iceberg_iceberg-spark-runtime-3.4_2.12-1.3.1.jar
2.5M	/home/jovyan/jars/org.projectnessie.nessie-integrations_nessie-spark-extensions-3.4_2.12-0.99.0.jar
12K	/home/jovyan/jars/org.reactivestreams_reactive-streams-1.0.4.jar
44K	/home/jovyan/jars/org.slf4j_slf4j-api-1.7.30.jar
16K	/home/jovyan/jars/software.amazon.awssdk_annotations-2.24.8.jar
533M	/home/jovyan/jars/software.amazon.awssdk_bundle-2.24.8.jar
84K	/home/jovyan/jars/software.amazon.awssdk_http-client-spi-2.24.8.jar
28K	/home/jovyan/jars/software.amazon.awssdk_metrics-spi-2.24.8.jar
36K	/home/jovyan/jars/software.amazon.awssdk_url-connection-client-2.24.8.jar
212K	/home/jovyan/jars/software.amazon.awssdk_utils-2.24.8.jar


In [10]:
spark.sql("select 1").show()

+---+
|  1|
+---+
|  1|
+---+



In [11]:
spark.sql("show catalogs").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



In [19]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [13]:
#-- 1. Création du schéma dans Nessie
spark.sql(f"""
CREATE NAMESPACE IF NOT EXISTS nessie.db_prod
""")
# - With data store in another folder
#spark.sql(f"""
#CREATE NAMESPACE IF NOT EXISTS nessie.db_prod 
#LOCATION '{NESSIE_DATA_STORE}'
#""")

DataFrame[]

In [14]:
spark.sql("DESCRIBE NAMESPACE nessie.db_prod").show()

+--------------+----------+
|     info_name|info_value|
+--------------+----------+
|  Catalog Name|    nessie|
|Namespace Name|   db_prod|
|         Owner|    jovyan|
+--------------+----------+



In [15]:
spark.sql("DESCRIBE NAMESPACE EXTENDED nessie.db_prod").show(truncate=False)

+--------------+----------+
|info_name     |info_value|
+--------------+----------+
|Catalog Name  |nessie    |
|Namespace Name|db_prod   |
|Owner         |jovyan    |
|Properties    |          |
+--------------+----------+



In [16]:
# Exemple : Création d'une table et insertion
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.db_prod.pyspark_test (
        id INT, 
        val STRING,
        event_date DATE
    )
    USING iceberg
    PARTITIONED BY (days(event_date))
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.metadata.compression-codec'='gzip',
        'write.parquet.compression-codec'='snappy',
        'write.parquet.row-group-size-bytes' = '33554432',
        'write.target-file-size-bytes' = '33554432',
        'write.metadata.metrics.default' = 'full'
    )
""")

DataFrame[]

In [17]:
# liste des tables dans le namespace
spark.sql("SHOW TABLES IN nessie.db_prod").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  db_prod|pyspark_test|      false|
+---------+------------+-----------+



In [18]:
# description d'une table
spark.sql("DESCRIBE TABLE nessie.db_prod.pyspark_test").show()

+--------------+----------------+-------+
|      col_name|       data_type|comment|
+--------------+----------------+-------+
|            id|             int|   null|
|           val|          string|   null|
|    event_date|            date|   null|
|              |                |       |
|# Partitioning|                |       |
|        Part 0|days(event_date)|       |
+--------------+----------------+-------+



In [19]:
# SPARK SQL
spark.sql("""
    INSERT INTO nessie.db_prod.pyspark_test 
    VALUES 
        (1, 'Donnée A', CAST('2026-10-01' AS DATE)),
        (2, 'Donnée B', CAST('2026-10-01' AS DATE)),
        (3, 'Donnée C', CAST('2026-10-02' AS DATE)),
        (4, 'Donnée D', CAST('2026-10-03' AS DATE))
""")

DataFrame[]

In [20]:
# SPARK DATAFRAME

from datetime import date
# Création d'une liste de données
data = [
    (5, "Donnée E", date(2026, 10, 4)),
    (6, "Donnée F", date(2026, 10, 5))
]

# Création du DataFrame avec le schéma correspondant
df = spark.createDataFrame(data, ["id", "val", "event_date"])

# Écriture dans la table Nessie (mode 'append')
df.writeTo("nessie.db_prod.pyspark_test").append()

In [21]:
# Lecture
spark.sql("SELECT * FROM nessie.db_prod.pyspark_test ORDER BY event_date").show()

+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
|  3|Donnée C|2026-10-02|
|  4|Donnée D|2026-10-03|
|  5|Donnée E|2026-10-04|
|  6|Donnée F|2026-10-05|
+---+--------+----------+



In [22]:
# pour une date
spark.sql("SELECT * FROM nessie.db_prod.pyspark_test WHERE event_date = '2026-10-01'").show()


+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
+---+--------+----------+



In [11]:
# pour une année
spark.sql("""
    SELECT * FROM nessie.db_prod.pyspark_test 
    WHERE year(event_date) = 2026
""").show()


+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  5|Donnée E|2026-10-04|
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
|  6|Donnée F|2026-10-05|
|  3|Donnée C|2026-10-02|
|  4|Donnée D|2026-10-03|
+---+--------+----------+



In [12]:
# pour un mois
spark.sql("""
    SELECT * FROM nessie.db_prod.pyspark_test 
    WHERE year(event_date) = 2026 
      AND month(event_date) = 10
""").show()


+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  5|Donnée E|2026-10-04|
|  6|Donnée F|2026-10-05|
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
|  3|Donnée C|2026-10-02|
|  4|Donnée D|2026-10-03|
+---+--------+----------+



In [13]:
# plage de date
spark.sql("""
    SELECT * FROM nessie.db_prod.pyspark_test 
    WHERE event_date BETWEEN '2026-10-01' AND '2026-10-15'
""").show()


+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
|  5|Donnée E|2026-10-04|
|  6|Donnée F|2026-10-05|
|  3|Donnée C|2026-10-02|
|  4|Donnée D|2026-10-03|
+---+--------+----------+



In [14]:
# fichiers dans minio
spark.sql("SELECT file_path, file_format, record_count FROM nessie.db_prod.pyspark_test.files").show(truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|file_path                                                                                                                                                                        |file_format|record_count|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|s3://nessie-bucket/warehouse/db_prod/pyspark_test_63a3852d-cd69-4571-921b-a1190ddc47a2/data/event_date_day=2026-10-04/00000-13-0eae1789-1d4f-4c23-9ced-281897ecf2b7-00001.parquet|PARQUET    |1           |
|s3://nessie-bucket/warehouse/db_prod/pyspark_test_63a3852d-cd69-4571-921b-a1190ddc47a2/data/event_date_day=2026-10-05/00000-13-0eae1789-1d4f-4c23-9ced-281897ecf2b7-00002.parquet|P

In [17]:
# stats
spark.sql("SELECT * FROM nessie.db_prod.pyspark_test.manifests")

DataFrame[content: int, path: string, length: bigint, partition_spec_id: int, added_snapshot_id: bigint, added_data_files_count: int, existing_data_files_count: int, deleted_data_files_count: int, added_delete_files_count: int, existing_delete_files_count: int, deleted_delete_files_count: int, partition_summaries: array<struct<contains_null:boolean,contains_nan:boolean,lower_bound:string,upper_bound:string>>]

In [19]:
# lister les partitions
spark.sql("SELECT * FROM nessie.db_prod.pyspark_test.partitions").show(truncate=False)


+------------+-------+------------+----------+----------------------------+--------------------------+----------------------------+--------------------------+
|partition   |spec_id|record_count|file_count|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|
+------------+-------+------------+----------+----------------------------+--------------------------+----------------------------+--------------------------+
|{2026-10-02}|0      |1           |1         |0                           |0                         |0                           |0                         |
|{2026-10-03}|0      |1           |1         |0                           |0                         |0                           |0                         |
|{2026-10-01}|0      |2           |1         |0                           |0                         |0                           |0                         |
|{2026-10-04}|0      |1           |1         |

In [15]:
# les snapshots
spark.sql("SELECT committed_at, snapshot_id, operation FROM nessie.db_prod.pyspark_test.snapshots").show()


+--------------------+-------------------+---------+
|        committed_at|        snapshot_id|operation|
+--------------------+-------------------+---------+
|2026-03-16 10:46:...|8671951644824558440|   append|
|2026-03-16 10:46:...|2657302441659340951|   append|
+--------------------+-------------------+---------+



In [16]:
# time travel
# Charger une version spécifique via Spark
ID_DU_SNAPSHOT="" # snapshots ID from list of snapshots
df_v1 = spark.read \
    .option("snapshot-id", ID_DU_SNAPSHOT) \
    .table("nessie.db_prod.pyspark_test")

df_v1.show()


+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
|  3|Donnée C|2026-10-02|
|  4|Donnée D|2026-10-03|
+---+--------+----------+



In [21]:
# RESTORE FROM SNAPSHOT ID (we are going to restore in new table)
ID_DU_SNAPSHOT = "8671951644824558440"

spark.sql(f"""
    CREATE TABLE nessie.db_prod.pyspark_test_restored 
    USING iceberg 
    AS SELECT * FROM nessie.db_prod.pyspark_test FOR SYSTEM_VERSION AS OF {ID_DU_SNAPSHOT}
""")

DataFrame[]

In [23]:
spark.sql("select * from nessie.db_prod.pyspark_test_restored ").show(10)

+---+--------+----------+
| id|     val|event_date|
+---+--------+----------+
|  1|Donnée A|2026-10-01|
|  2|Donnée B|2026-10-01|
|  3|Donnée C|2026-10-02|
|  4|Donnée D|2026-10-03|
+---+--------+----------+



In [23]:
spark.stop()